In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns


In [2]:
ratings_df    = pd.read_csv(r'd:\BookRecommendation\data\processed\clean_book_ratings.csv')
user_features = pd.read_csv(r'd:\BookRecommendation\data\processed\user_features.csv')
book_features = pd.read_csv(r'd:\BookRecommendation\data\processed\book_features.csv')
user_clusters = pd.read_csv(r'd:\BookRecommendation\data\processed\user_clusters.csv')
book_clusters = pd.read_csv(r'd:\BookRecommendation\data\processed\book_clusters.csv')
cf_scores_df  = pd.read_csv(r'd:\BookRecommendation\data\processed\cf_scores.csv')

In [3]:
import joblib

In [4]:
xgb_model    = joblib.load(r'd:\BookRecommendation\ml_models\xgb_model.pkl')
preprocessor = joblib.load(r'd:\BookRecommendation\ml_models\preprocessor.pkl')
svd_model    = joblib.load(r'd:\BookRecommendation\ml_models\cf_svd_model.pkl')

In [23]:
book_features.columns

Index(['ISBN', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std',
       'Year_Of_Publication', 'Publisher', 'Genre', 'Title_Author_combined',
       'Genre_grouped'],
      dtype='str')

In [6]:
# Start with cf_scores_df (15.6L rows)
rec_df = cf_scores_df.copy()
print(f"Base shape: {rec_df.shape}")

# Merge user features
user_cols = ['User_ID', 'user_avg_rating', 'user_rating_count',
             'user_rating_std', 'old_books_ratio', 'recent_books_ratio',
             'genre_diversity', 'favourite_author','Age_Group']

rec_df = rec_df.merge(user_features[user_cols], on='User_ID', how='left')
print(f"After user features: {rec_df.shape}")

# Merge user cluster
rec_df = rec_df.merge(
    user_clusters[['User_ID', 'user_cluster']], 
    on='User_ID', how='left'
)
print(f"After user cluster: {rec_df.shape}")

# Merge book features
book_cols = ['ISBN', 'Book_avg_rating', 'Book_rating_count',
             'Book_rating_std', 'Year_Of_Publication', 
             'Genre_grouped']
rec_df = rec_df.merge(book_features[book_cols], on='ISBN', how='left')
print(f"After book features: {rec_df.shape}")

# Merge book cluster
rec_df = rec_df.merge(
    book_clusters[['ISBN', 'book_cluster']], 
    on='ISBN', how='left'
)
print(f"After book cluster: {rec_df.shape}")

# Check
print(f"\nColumns: {rec_df.columns.tolist()}")
print(f"Missing values:\n{rec_df.isnull().sum()}")

Base shape: (1561400, 3)
After user features: (1561400, 11)
After user cluster: (1561400, 12)
After book features: (1561400, 17)
After book cluster: (1561400, 18)

Columns: ['User_ID', 'ISBN', 'cf_predicted_score', 'user_avg_rating', 'user_rating_count', 'user_rating_std', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity', 'favourite_author', 'Age_Group', 'user_cluster', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'Genre_grouped', 'book_cluster']
Missing values:
User_ID                0
ISBN                   0
cf_predicted_score     0
user_avg_rating        0
user_rating_count      0
user_rating_std        0
old_books_ratio        0
recent_books_ratio     0
genre_diversity        0
favourite_author       0
Age_Group              0
user_cluster           0
Book_avg_rating        0
Book_rating_count      0
Book_rating_std        0
Year_Of_Publication    0
Genre_grouped          0
book_cluster           0
dtype: int64


In [7]:
# Get Book_Author from ratings_df
book_author = ratings_df[['ISBN', 'Book_Author']].drop_duplicates()
rec_df = rec_df.merge(book_author, on='ISBN', how='left')

print(f"Shape after adding Book_Author: {rec_df.shape}")
print(f"Missing Book_Author: {rec_df['Book_Author'].isnull().sum()}")

Shape after adding Book_Author: (1561400, 19)
Missing Book_Author: 0


In [8]:

# Author match
rec_df['author_match'] = (
    rec_df['favourite_author'] == rec_df['Book_Author']
).astype(int)

# Popularity signal
median_count = book_features['Book_rating_count'].median()
rec_df['is_popular'] = (
    rec_df['Book_rating_count'] > median_count
).astype(int)

# Implicit interaction signal
implicit = ratings_df[ratings_df['Book_Rating'] == 0][
    ['User_ID', 'ISBN']
].copy()
implicit['interacted'] = 1

rec_df = rec_df.merge(implicit, on=['User_ID', 'ISBN'], how='left')
rec_df['interacted'] = rec_df['interacted'].fillna(0).astype(int)

print(f"Shape after interaction features: {rec_df.shape}")
print(f"Columns: {rec_df.columns.tolist()}")

Shape after interaction features: (1561400, 22)
Columns: ['User_ID', 'ISBN', 'cf_predicted_score', 'user_avg_rating', 'user_rating_count', 'user_rating_std', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity', 'favourite_author', 'Age_Group', 'user_cluster', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'Genre_grouped', 'book_cluster', 'Book_Author', 'author_match', 'is_popular', 'interacted']


In [9]:
# Keep ISBN and User_ID separate for later
rec_user_ids = rec_df['User_ID']
rec_isbns    = rec_df['ISBN']

# Drop columns not needed for prediction
rec_features = rec_df.drop(columns=[
    'User_ID', 'ISBN', 
    'favourite_author', 'Book_Author',
    'cf_predicted_score'
])

# Rename cf_predicted_score to match training
rec_features['cf_predicted_score'] = rec_df['cf_predicted_score']

print(f"Features shape: {rec_features.shape}")
print(f"Columns: {rec_features.columns.tolist()}")

Features shape: (1561400, 18)
Columns: ['user_avg_rating', 'user_rating_count', 'user_rating_std', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity', 'Age_Group', 'user_cluster', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'Genre_grouped', 'book_cluster', 'author_match', 'is_popular', 'interacted', 'cf_predicted_score']


In [10]:
# Define categorical and numeric features
# Must be EXACTLY same as training in Notebook 04

categorical_features = ['Age_Group', 'Genre_grouped']

numeric_features = [col for col in rec_features.columns
                   if col not in categorical_features]

print(f"Categorical features: {categorical_features}")
print(f"Numeric features: {numeric_features}")
print(f"Total features: {len(categorical_features) + len(numeric_features)}")

Categorical features: ['Age_Group', 'Genre_grouped']
Numeric features: ['user_avg_rating', 'user_rating_count', 'user_rating_std', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity', 'user_cluster', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'book_cluster', 'author_match', 'is_popular', 'interacted', 'cf_predicted_score']
Total features: 18


In [12]:
# Apply same preprocessor used in training
rec_processed = preprocessor.transform(rec_features)

# Get feature names
ohe_feature_names = preprocessor.named_transformers_['OneHotEncoding'].get_feature_names_out(categorical_features)
all_feature_names = list(ohe_feature_names) + list(numeric_features)

# Convert to dataframe
rec_processed = pd.DataFrame(rec_processed, columns=all_feature_names)

print(f"Processed shape: {rec_processed.shape}")

Processed shape: (1561400, 35)


In [14]:
rec_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 1561400 entries, 0 to 1561399
Data columns (total 35 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   Age_Group_Middle_Age            1561400 non-null  float64
 1   Age_Group_Senior                1561400 non-null  float64
 2   Age_Group_Teen                  1561400 non-null  float64
 3   Age_Group_Unknown               1561400 non-null  float64
 4   Age_Group_Young_Adult           1561400 non-null  float64
 5   Genre_grouped_Children          1561400 non-null  float64
 6   Genre_grouped_Drama_Humor       1561400 non-null  float64
 7   Genre_grouped_Fiction           1561400 non-null  float64
 8   Genre_grouped_Historical        1561400 non-null  float64
 9   Genre_grouped_Horror            1561400 non-null  float64
 10  Genre_grouped_Non_Fiction       1561400 non-null  float64
 11  Genre_grouped_Other             1561400 non-null  float64
 12  Genre_group

In [15]:
# Get exact column order from trained model
expected_columns = xgb_model.get_booster().feature_names

print(f"Expected columns: {expected_columns}")
print(f"Your columns: {rec_processed.columns.tolist()}")

# Reorder rec_processed columns to match exactly
rec_processed = rec_processed[expected_columns]

print(f"Shape after reordering: {rec_processed.shape}")

Expected columns: ['Age_Group_Middle_Age', 'Age_Group_Senior', 'Age_Group_Teen', 'Age_Group_Unknown', 'Age_Group_Young_Adult', 'Genre_grouped_Children', 'Genre_grouped_Drama_Humor', 'Genre_grouped_Fiction', 'Genre_grouped_Historical', 'Genre_grouped_Horror', 'Genre_grouped_Non_Fiction', 'Genre_grouped_Other', 'Genre_grouped_Religious', 'Genre_grouped_Romance', 'Genre_grouped_SciFi_Fantasy', 'Genre_grouped_Thriller_Mystery', 'Genre_grouped_Unknown', 'Genre_grouped_War_Adventure', 'Genre_grouped_Young_Adult', 'user_avg_rating', 'user_rating_count', 'user_rating_std', 'old_books_ratio', 'recent_books_ratio', 'genre_diversity', 'user_cluster', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std', 'Year_Of_Publication', 'book_cluster', 'cf_predicted_score', 'author_match', 'is_popular', 'interacted']
Your columns: ['Age_Group_Middle_Age', 'Age_Group_Senior', 'Age_Group_Teen', 'Age_Group_Unknown', 'Age_Group_Young_Adult', 'Genre_grouped_Children', 'Genre_grouped_Drama_Humor', 'Genre_gro

In [16]:
# Now predict
rec_df['xgb_predicted_score'] = xgb_model.predict(rec_processed)

print("Predictions generated!")
print(rec_df['xgb_predicted_score'].describe())

Predictions generated!
count    1.561400e+06
mean     1.470988e+00
std      1.099049e+00
min     -1.287114e+00
25%      8.507959e-01
50%      1.238109e+00
75%      1.679371e+00
max      8.072781e+00
Name: xgb_predicted_score, dtype: float64


In [17]:
# Add User_ID and ISBN back
rec_df['User_ID'] = rec_user_ids.values
rec_df['ISBN']    = rec_isbns.values

# Sort by predicted score and get top 10 per user
top_10_recommendations = (rec_df
    .sort_values('xgb_predicted_score', ascending=False)
    .groupby('User_ID')
    .head(10)
    .reset_index(drop=True)
)

print(f"Total recommendation rows: {top_10_recommendations.shape}")
print(top_10_recommendations[['User_ID', 'ISBN', 
                               'xgb_predicted_score']].head(20))

Total recommendation rows: (78070, 23)
    User_ID        ISBN  xgb_predicted_score
0    146057  0971880107             8.072781
1    171079  0971880107             7.866910
2    114085  0425182908             7.818035
3    132846  080411868X             7.594503
4    171846  0971880107             7.538657
5    249953  0425182908             7.531864
6      1248  0060977493             7.529378
7      1248  0671042858             7.477643
8    169558  0971880107             7.460587
9      1248  0440236673             7.450005
10     1248  0767905180             7.447357
11   171079  0140244824             7.415590
12   146057  0375703055             7.412507
13    35233  0440222656             7.394377
14   151370  0425182908             7.381718
15   132846  006101351X             7.381443
16   249953  0140244824             7.372496
17    15589  0971880107             7.322739
18   249953  0515126772             7.321960
19   114085  0425175405             7.321949


In [22]:
# Sort by User_ID first, then by score descending
top_10_recommendations = (rec_df
    .sort_values(['User_ID', 'xgb_predicted_score'], 
                 ascending=[True, False])
    .groupby('User_ID')
    .head(10)
    .reset_index(drop=True)
)

# Add book info
book_info = ratings_df[['ISBN', 'Book_Title', 
                         'Book_Author']].drop_duplicates()

top_10_recommendations = top_10_recommendations.merge(
    book_info, on='ISBN', how='left'
)

# Check recommendations for a specific user
user_id = int(input("please enter the user id")) # example user

user_recs = top_10_recommendations[
    top_10_recommendations['User_ID'] == user_id
][['Book_Title', 'xgb_predicted_score']]

print(f"Top 10 recommendations for User {user_id}:")
print(user_recs.to_string(index=False))

Top 10 recommendations for User 1248:
                                                           Book_Title  xgb_predicted_score
                                              The God of Small Things             7.529378
                                        The Girl Who Loved Tom Gordon             7.477643
                                                         The Brethren             7.450005
                     Jemima J: A Novel About Ugly Ducklings and Swans             7.447357
Three To Get Deadly : A Stephanie Plum Novel (A Stephanie Plum Novel)             7.198496
                                                               Sphere             7.151244
                                                  Angels &amp; Demons             7.150781
                                                           Night Sins             7.148212
                    All the Pretty Horses (The Border Trilogy, Vol 1)             7.098220
                                                    